# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [ ]:
# Task 1 — Treemap: fossil fuel dependency by country

# ── Data prep: fossil sources only ────────────────────────────────────────────
df_fossil = df.loc[df['Source_Type'] == 'Fossil']

# CVD-safe mapping for the three fossil sources.
# Coal keeps this colour in Task 3, where it is the accent on the stacked bar.
fossil_color_map = {
    'Coal':        '#E07B39',   # orange
    'Oil':         '#2E75B6',   # blue
    'Natural Gas': '#F5C842',   # yellow
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',                       # TWh sums correctly at every level
    color='Source',
    color_discrete_map=fossil_color_map,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white',
)

# ── Step 2: Customisation — TWh in labels, no percentages ─────────────────────
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)

fig.update_layout(
    title=('<span style="color:#E07B39"><b>Coal</b></span> drives '
           "Asia's fossil energy — China burns far more fossil TWh than any other country"),
    font=dict(family='Arial', size=12),
)

# Grey out parent nodes (Region and Country) — keep only the three source colours
fig.data[0].marker.colors = [
    c if c in fossil_color_map.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [ ]:
# Task 2 — charts: low-carbon energy (Nuclear + Hydro) per country

# ── Data prep: low-carbon only, aggregated by country ─────────────────────────
low_carbon = (df.loc[df['Source_Type'] == 'Low-carbon']
                .groupby('Country')['TWh'].sum()
                .reset_index())

# Leading country — used in the bar-chart insight title
leader = low_carbon.sort_values('TWh', ascending=False).iloc[0]

# ── Chart A: treemap, single level with a dummy root ──────────────────────────
fig_tree = px.treemap(
    low_carbon,
    path=[px.Constant('Low-carbon'), 'Country'],     # dummy root node labelled 'Low-carbon'
    values='TWh',
    color_discrete_sequence=['#2E75B6'],             # single CVD-safe hue
    height=600, width=1000,
    template='plotly_white',
)
fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)
fig_tree.update_layout(
    title='Low-carbon energy (Nuclear + Hydro) by country',
    font=dict(family='Arial', size=12),
)
fig_tree.show()

# ── Chart B: horizontal bar, sorted by TWh ────────────────────────────────────
country_order = low_carbon.sort_values('TWh', ascending=True)['Country'].tolist()  # asc → largest on top

fig_bar = px.bar(
    low_carbon,
    x='TWh', y='Country',
    orientation='h',
    text=low_carbon['TWh'].map('{:,.0f}'.format),
    color_discrete_sequence=['#2E75B6'],             # same CVD-safe blue
    category_orders={'Country': country_order},
    labels={'TWh': 'Low-carbon energy (TWh)'},
    height=650, width=900,
    template='plotly_white',
)
fig_bar.update_traces(textposition='outside', cliponaxis=False)
fig_bar.update_layout(
    title=f'{leader["Country"]} leads the world in low-carbon energy '
          f'with {leader["TWh"]:,.0f} TWh from nuclear and hydro',
    font=dict(family='Arial', size=12),
    yaxis=dict(title=''),
)
fig_bar.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The **horizontal bar chart**. All the bars share a common baseline, so the reader judges rank by comparing *lengths* along a single axis — an easy, precise visual task. The treemap encodes the same values as rectangle *areas* with no shared edge, and area is much harder to compare accurately, especially for the smaller countries near the bottom of the ranking. Sorting the bars makes the order explicit and the gaps between countries readable, whereas in the treemap the tiles are packed by a layout algorithm and their positions carry no ranking meaning. The treemap would only win if the hierarchy or part-to-whole composition were the story — here it is a plain one-level ranking, which is exactly what bars are best at.


## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [ ]:
# Task 3 — 100% stacked bar: coal share of fossil energy by region
# Same fossil data as the Task 1 treemap, visualised a second way.

# ── Data prep: reuse df_fossil, aggregate by Region × Source, convert to shares
region_mix = (df_fossil.groupby(['Region', 'Source'])['TWh'].sum().reset_index())
region_mix['Share'] = region_mix['TWh'] / region_mix.groupby('Region')['TWh'].transform('sum')

# Sort regions by coal share, most coal-dependent at the top
region_order = (region_mix.loc[region_mix['Source'] == 'Coal']
                          .sort_values('Share', ascending=False)['Region'].tolist())

# Grey + one accent: coal keeps the Task 1 orange, oil & gas become two greys
coal_accent_map = {
    'Coal':        '#E07B39',   # orange accent — the story (same as Task 1)
    'Oil':         '#BBBBBB',   # mid grey — context
    'Natural Gas': '#DDDDDD',   # light grey — context
}

# Percentage labels on the coal segments only; empty everywhere else
region_mix['Label'] = np.where(region_mix['Source'] == 'Coal',
                               region_mix['Share'].map('{:.0%}'.format), '')

# Most coal-dependent region, for the insight title
top = (region_mix.loc[region_mix['Source'] == 'Coal']
                 .sort_values('Share', ascending=False).iloc[0])

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.bar(
    region_mix,
    x='Share', y='Region',
    color='Source',
    text='Label',
    color_discrete_map=coal_accent_map,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal first → on the baseline
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_name='Region',
    hover_data={'Label': False, 'Share': ':.1%'},
    height=600, width=1000,
    template='plotly_white',
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=11),
    marker_line_color='white', marker_line_width=1,
)

fig.update_layout(
    title=f'{top["Region"]} is the most coal-dependent region — '
          f'coal is {top["Share"]:.0%} of its fossil energy',
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.04),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
)

# Mark the end of the scale — every bar sums to 100%
fig.add_annotation(
    x=1, y=0, xref='x', yref='paper',
    xanchor='right', yanchor='top', yshift=-4,
    text='100%', showarrow=False,
    font=dict(size=11, color='grey'),
)

fig.show()


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- **Only the treemap answers:** *How large is each country's (and region's) fossil consumption in absolute terms?* Area is proportional to TWh, so you can see that China's coal tile dwarfs everything else and compare the actual magnitudes across the full Region → Country → Source hierarchy. The 100% stacked bar normalises every region to 100%, throwing away all size information — a tiny region and a huge one look identical.
- **Only the stacked bar answers:** *Within its own fossil mix, which region leans hardest on coal?* Because each bar is rescaled to shares and sorted with coal on a common baseline, you can rank regions by coal *proportion* at a glance. The treemap mixes absolute size with nesting, so comparing the coal *share* of one region against another means eyeballing ratios of areas across separate branches — which it effectively hides.
